<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_V2_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03-V2: Optimización del Baseline - Random Forest (Balanceado)
**Proyecto:** Detección de deslizamientos mediante IA | Universidad EAFIT

**Cambios clave en esta versión:**
1. **Manejo de Desbalance:** Uso de `class_weight='balanced_subsample'`.
2. **Resolución HOG:** Ajuste a 8x8 para detectar deslizamientos de menor escala.
3. **Evaluación:** Curva ROC, AUC y Matriz de Confusión unificadas.

In [ ]:
from google.colab import drive
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir = img_dirs[0]
    train_mask_dir = train_img_dir.parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ {len(img_list)} muestras detectadas para la V2.")
else:
    print("❌ ERROR: Ruta no encontrada en Drive.")

In [ ]:
N_SAMPLES = 1000 
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc="Extrayendo HOG 8x8"):
    with h5py.File(img_list[i], "r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], "r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    
    rgb = patch[:,:,[3,2,1]]
    # Parche NumPy 2.0
    rgb_norm = ((rgb - np.min(rgb)) / (np.ptp(rgb) + 1e-8) * 255).astype("uint8")
    
    # HOG con mayor resolución
    feats_hog = hog(rgb_norm, pixels_per_cell=(8,8), cells_per_block=(2,2), channel_axis=-1, feature_vector=True)
    avg_slope = np.mean(patch[:,:,12])
    
    X.append(np.hstack([feats_hog, avg_slope]))
    y.append(int(np.max(mask) > 0))

X, y = np.array(X), np.array(y)
print(f"✓ Dataset listo: {X.shape}")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(
        n_estimators=200, 
        n_jobs=-1, 
        random_state=42, 
        class_weight='balanced_subsample'
    )
    rf.fit(X[t_idx], y[t_idx])
    preds = rf.predict(X[v_idx])
    f1_scores.append(f1_score(y[v_idx], preds))

# Estadísticas Finales
y_probs = rf.predict_proba(X[v_idx])[:, 1]
fpr, tpr, _ = roc_curve(y[v_idx], y_probs)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ConfusionMatrixDisplay(confusion_matrix(y[v_idx], preds)).plot(cmap='YlGnBu', ax=ax[0])
ax[1].plot(fpr, tpr, color='darkorange', label=f'AUC = {roc_auc:.3f}')
ax[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
ax[1].set_title("Curva ROC Balanceada")
ax[1].legend()
plt.show()

print(f"🚀 F1-Score Promedio: {np.mean(f1_scores):.4f}")
print(f"🔥 AUC Resultante: {roc_auc:.4f}")